# ZHAW Commute Survey Simulator

Simulates answers to the ZHAW commute survey and saves them as `zhaw_commute_responses.csv`, in exactly the same format as the CSV export of the survey app (`/export.csv`). Use it to practise the analysis before the real class data is available.

The answers are not purely random. The simulation follows a simple story:
1. Where you live determines how far you are from ZHAW (students in the category *Other*, e.g. living with their parents, live further away). Gender is simulated independently on purpose.
2. The distance determines the means of transport (short: on foot or by bike, long: train or car).
3. Distance and means of transport determine the travel time and the monthly costs.
4. Travel time and means of transport influence how satisfied and stressed you are, and whether you can use the time productively.

## Libraries and settings

In [1]:
# Libraries
import os
import numpy as np
import pandas as pd

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Show current working directory
print(os.getcwd())

/workspaces/data_analytics/Week_LC/LC_07/zhaw_commute_survey/analysis


## Random seed
Use your own birthday, so your simulated data is unique to you.

In [2]:
day, month = 15, 6  # <-- replace with your own birth day and month
rng = np.random.default_rng(day * month)

n = 200  # number of simulated respondents

## Helper functions
- `choose()` draws one option per respondent from row-specific probabilities, so each respondent can have different probabilities.
- `likert()` turns a continuous latent value into a 1..5 answer by cutting it at fixed thresholds.

In [3]:
def choose(options, probs):
    probs = np.asarray(probs, dtype=float)
    probs = probs / probs.sum(axis=1, keepdims=True)
    u = rng.random((len(probs), 1))
    idx = (u > probs.cumsum(axis=1)).sum(axis=1)
    return np.array(options)[np.minimum(idx, len(options) - 1)]


def likert(latent, noise=0.8):
    x = latent + rng.normal(0, noise, len(latent))
    return np.digitize(x, [-1.5, -0.5, 0.5, 1.5]) + 1

## Campus, gender, age, living situation and distance

In [4]:
campus = rng.choice(['Winterthur', 'Zürich', 'Wädenswil'], n, p=[0.50, 0.35, 0.15])
gender = rng.choice(['Female', 'Male'], n, p=[0.48, 0.52])
age = np.clip(np.round(19 + rng.gamma(2.0, 2.0, n)), 18, 45).astype(int)

# Older students more often live in their own flat
p_own = np.clip(0.10 + 0.04 * (age - 19), 0.10, 0.70)
living = choose(['Own flat', 'Shared apartment', 'Other'], np.column_stack([p_own, 0.35 * np.ones(n), 0.55 - p_own + 0.10]))

# Median distance (km) depends on the living situation; distances are right-skewed (log-normal)
median_km = pd.Series(living).map({'Own flat': 9, 'Shared apartment': 4, 'Other': 22}).values
distance_km = np.clip(np.round(np.exp(rng.normal(np.log(median_km), 0.8))), 0, 150).astype(int)

## Means of transport
Short distances: on foot or by bike. Long distances: train or car. More cars at the Wädenswil campus.

In [5]:
transport_options = ['Train', 'Tram or bus', 'Car or motorbike', 'Bike or e-bike', 'On foot']
probs = np.select(
    [distance_km[:, None] < 2, distance_km[:, None] < 8],
    [np.array([0.00, 0.15, 0.00, 0.45, 0.40]),
     np.array([0.10, 0.45, 0.10, 0.35, 0.00])],
    np.array([0.75, 0.05, 0.20, 0.00, 0.00]))
probs = probs * np.where((campus == 'Wädenswil')[:, None], [1, 0.7, 2.5, 1, 1], 1)
transport = choose(transport_options, probs)

## Travel time door to door
Speed (km/h) and fixed extra minutes (walking to the station, waiting, parking) per means of transport, plus random variation.

In [6]:
speed = {'Train': 45, 'Tram or bus': 15, 'Car or motorbike': 40, 'Bike or e-bike': 15, 'On foot': 5}
extra_min = {'Train': 20, 'Tram or bus': 8, 'Car or motorbike': 10, 'Bike or e-bike': 3, 'On foot': 0}

t = pd.Series(transport)
minutes = (np.maximum(distance_km, 0.5) / t.map(speed).values * 60 + t.map(extra_min).values)
door_to_door_min = np.clip(np.round(minutes * np.exp(rng.normal(0, 0.2, n))), 3, 240).astype(int)

days_per_week = rng.choice([2, 3, 4, 5], n, p=[0.10, 0.25, 0.35, 0.30])

## Travel pass and monthly costs (CHF)

In [7]:
pass_options = ['GA travelcard', 'Regional pass (e.g. ZVV)', 'Half Fare travelcard only', 'No pass']
pass_probs = {
    'Train':            [0.55, 0.35, 0.08, 0.02],
    'Tram or bus':      [0.15, 0.70, 0.10, 0.05],
    'Car or motorbike': [0.10, 0.10, 0.40, 0.40],
    'Bike or e-bike':   [0.05, 0.15, 0.30, 0.50],
    'On foot':          [0.05, 0.15, 0.30, 0.50],
}
travel_pass = choose(pass_options, [pass_probs[m] for m in transport])

trips_per_month = days_per_week * 2 * 4.3
cost = np.select(
    [transport == 'Car or motorbike',
     np.isin(transport, ['Bike or e-bike', 'On foot']),
     travel_pass == 'GA travelcard',
     travel_pass == 'Regional pass (e.g. ZVV)',
     travel_pass == 'Half Fare travelcard only'],
    [0.25 * distance_km * trips_per_month + 60,      # fuel + parking
     np.where(transport == 'On foot', 0, 15),        # bike maintenance
     220,                                            # GA travelcard for young people, per month
     np.minimum(50 + 3 * distance_km, 150),          # regional pass, more zones for longer trips
     0.15 * distance_km * trips_per_month + 15],     # tickets with Half Fare + share of the card
    0.30 * distance_km * trips_per_month)            # full-price tickets
monthly_cost_chf = np.clip(np.round(cost * np.exp(rng.normal(0, 0.15, n)) / 5) * 5, 0, 2000).astype(int)

## Likert answers
Longer travel time: less satisfied, more stressed. Train: time can be used productively. Car: stressful.

In [8]:
active = np.isin(transport, ['Bike or e-bike', 'On foot'])
productive_effect = t.map({'Train': 1.2, 'Tram or bus': 0.2, 'Car or motorbike': -1.5,
                           'Bike or e-bike': -1.2, 'On foot': -0.8}).values

satisfied = likert(0.8 - 0.025 * (door_to_door_min - 35) + 0.4 * active + 0.2 * productive_effect)
stressful = likert(-0.4 + 0.02 * (door_to_door_min - 35) + 0.8 * (transport == 'Car or motorbike') - 0.4 * active)
productive = likert(productive_effect + 0.01 * (door_to_door_min - 35))

## Assemble the data set
Same columns and order as the export of the survey app.

In [9]:
created_at = pd.Timestamp('2026-10-28 10:15', tz='UTC') + pd.to_timedelta(np.sort(rng.uniform(0, 10 * 60, n)), unit='s')

df = pd.DataFrame({
    'id': np.arange(1, n + 1),
    'created_at': created_at.map(lambda ts: ts.isoformat()),
    'campus': campus,
    'transport': transport,
    'door_to_door_min': door_to_door_min,
    'distance_km': distance_km,
    'days_per_week': days_per_week,
    'monthly_cost_chf': monthly_cost_chf,
    'travel_pass': travel_pass,
    'satisfied': satisfied,
    'stressful': stressful,
    'productive': productive,
    'gender': gender,
    'age': age,
    'living': living,
})
df.head(10)

,id,created_at,campus,transport,door_to_door_min,distance_km,days_per_week,monthly_cost_chf,travel_pass,satisfied,stressful,productive,gender,age,living
0,1,2026-10-28T10:15:00.451910954+00:00,Winterthur,Bike or e-bike,6,1,5,15,No pass,5,3,1,Male,21,Shared apartment
1,2,2026-10-28T10:15:10.761761321+00:00,Zürich,Train,25,8,4,185,GA travelcard,4,3,5,Male,31,Shared apartment
2,3,2026-10-28T10:15:12.944972746+00:00,Zürich,Train,39,14,5,240,GA travelcard,5,5,4,Male,21,Other
3,4,2026-10-28T10:15:13.454496419+00:00,Winterthur,Train,215,150,4,180,GA travelcard,1,5,5,Female,22,Other
4,5,2026-10-28T10:15:14.271052016+00:00,Zürich,Tram or bus,40,6,5,85,No pass,3,3,4,Female,25,Shared apartment
5,6,2026-10-28T10:15:20.119325680+00:00,Winterthur,Train,21,6,2,70,Regional pass (e.g. ZVV),4,2,5,Female,20,Shared apartment
6,7,2026-10-28T10:15:21.679335160+00:00,Zürich,Bike or e-bike,11,2,3,15,No pass,4,2,1,Male,22,Shared apartment
7,8,2026-10-28T10:15:21.808083556+00:00,Wädenswil,Train,45,17,3,210,GA travelcard,2,3,3,Female,20,Shared apartment
8,9,2026-10-28T10:15:22.112758254+00:00,Winterthur,Train,22,8,3,80,Regional pass (e.g. ZVV),3,4,4,Male,23,Own flat
9,10,2026-10-28T10:15:28.766268461+00:00,Winterthur,Bike or e-bike,13,4,4,15,No pass,5,3,1,Female,20,Own flat


In [10]:
# Overview
print(df.shape)
df.groupby('transport')[['door_to_door_min', 'distance_km', 'monthly_cost_chf', 'satisfied']].median()

(200, 15)


,door_to_door_min,distance_km,monthly_cost_chf,satisfied
transport,,,,
Bike or e-bike,19.0,4.0,15.0,4.0
Car or motorbike,41.0,20.0,215.0,3.0
Train,43.5,18.0,180.0,4.0
Tram or bus,25.5,4.5,60.0,4.0


## Save the simulated data
The analysis notebook reads the data from this file.

In [11]:
df.to_csv('zhaw_commute_responses.csv', index=False)
print('Saved', len(df), 'rows to zhaw_commute_responses.csv')

Saved 200 rows to zhaw_commute_responses.csv


### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [12]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-21 08:46:31
Python Version: 3.11.16
-----------------------------------
